# Parallelization Pattern (Scatter-Gather)
### When waiting is the bottleneck

Some process steps require analyzing multiple independent inputs before reaching a conclusion. Running them sequentially creates unnecessary delays. The parallelization pattern fans out independent tasks to multiple agents simultaneously, then aggregates the results — turning sequential bottlenecks into parallel execution.

Signals your process needs parallelization:
1. **Multiple independent inputs** must be analyzed before reaching a conclusion
2. **Sequential processing creates unacceptable time-to-decision**
3. **Synthesis requires reasoning** — results must be weighed, not just combined

### Business Use Case: AnyComp Telecom Quarterly Network Audit

5 regions assessed simultaneously. Each assessment is independent. The aggregator synthesizes into a prioritized remediation plan.

### Architecture

```
                         Orchestrator
                              │
              ┌──────┬────────┼────────┬──────┐
              ▼      ▼        ▼        ▼      ▼
           North  South    East     West  Central
           Agent  Agent    Agent    Agent  Agent
              │      │        │        │      │
              └──────┴────────┼────────┴──────┘
                              ▼
                         Aggregator → Prioritized action plan

Sequential: ~25 sec    Parallel: ~5 sec + synthesis
```




## Install Dependencies

Run this cell first, then **restart the kernel** and continue.

In [ ]:
!pip install -q strands-agents>=1.36.0 strands-agents-tools boto3 "typing_extensions>=4.12.0"

## Install Dependencies

**Restart the kernel** after installing, then continue.

## Setup: Regional Network Data

In [ ]:
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

REGION = boto3.session.Session().region_name or "us-east-1"

REGIONAL_DATA = {
    "North": {
        "region": "North",
        "uptime_pct": 99.2,
        "incidents": 12,
        "critical_incidents": 2,
        "avg_latency_ms": 18,
        "capacity_utilization_pct": 72,
        "subscriber_count": 145000,
        "notes": "Two critical fiber cuts in Q1. Redundant paths held but capacity was reduced."
    },
    "South": {
        "region": "South",
        "uptime_pct": 99.8,
        "incidents": 4,
        "critical_incidents": 0,
        "avg_latency_ms": 12,
        "capacity_utilization_pct": 58,
        "subscriber_count": 98000,
        "notes": "Stable quarter. Firmware updates completed on schedule."
    },
    "East": {
        "region": "East",
        "uptime_pct": 98.5,
        "incidents": 23,
        "critical_incidents": 5,
        "avg_latency_ms": 35,
        "capacity_utilization_pct": 91,
        "subscriber_count": 210000,
        "notes": "Highest incident count. Capacity near limit. Three tower outages due to weather."
    },
    "West": {
        "region": "West",
        "uptime_pct": 99.6,
        "incidents": 7,
        "critical_incidents": 1,
        "avg_latency_ms": 15,
        "capacity_utilization_pct": 65,
        "subscriber_count": 132000,
        "notes": "One critical incident: core router failure. Resolved in 4 hours."
    },
    "Central": {
        "region": "Central",
        "uptime_pct": 99.4,
        "incidents": 9,
        "critical_incidents": 1,
        "avg_latency_ms": 22,
        "capacity_utilization_pct": 78,
        "subscriber_count": 167000,
        "notes": "Capacity trending up. Planning expansion for Q3."
    },
}

print(f'{len(REGIONAL_DATA)} regions loaded')

## Strands Implementation

We use `ThreadPoolExecutor` to scatter regional assessments across 5 parallel agent invocations, then an aggregator agent synthesizes the results into a prioritized action plan.

In [ ]:
from strands import Agent
from strands.models import BedrockModel

model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    region_name=REGION,
    max_tokens=4096,
)

def assess_region(region_name, region_data):
    """Run a regional assessment agent. Called in parallel."""
    agent = Agent(
        model=model,
        system_prompt=(
            "You are a network auditor for AnyComp Telecom. "
            "Assess the given regional network data and produce a brief audit finding: "
            "status (healthy/at-risk/critical), key issues, and recommended actions. "
            "Be concise — 3-5 sentences max."
        ),
    )
    result = agent(f'Assess the {region_name} region:\n{json.dumps(region_data, indent=2)}')
    return region_name, str(result)


# ── Scatter: parallel regional assessments ────────────────────────────────────
print("=" * 60)
print("STRANDS — Parallel Regional Assessments")
print("=" * 60)

start = time.time()
assessments = {}

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {
        executor.submit(assess_region, name, data): name
        for name, data in REGIONAL_DATA.items()
    }
    for future in as_completed(futures):
        region_name, result = future.result()
        assessments[region_name] = result
        print(f'  ✅ {region_name} assessed')

parallel_time = time.time() - start
print(f'\nAll 5 regions assessed in {parallel_time:.1f}s (parallel)')

# ── Gather: aggregator synthesizes ───────────────────────────────────────────
aggregator = Agent(
    model=model,
    system_prompt=(
        "You are the Chief Network Officer at AnyComp Telecom. "
        "Given audit findings from all 5 regions, produce a quarterly audit summary: "
        "1) Overall network health rating, 2) Top 3 issues ranked by severity, "
        "3) Prioritized remediation plan with owners and timelines."
    ),
)

all_findings = '\n\n'.join(f'=== {r} ===\n{a}' for r, a in assessments.items())
print()
print("Aggregator synthesizing...")
summary = aggregator(f'Regional audit findings:\n\n{all_findings}')
print()
print(summary)

### Compare: Sequential vs Parallel Timing

In [ ]:
# Run sequentially for comparison
start_seq = time.time()
for name, data in REGIONAL_DATA.items():
    assess_region(name, data)
sequential_time = time.time() - start_seq

print(f'Sequential: {sequential_time:.1f}s')
print(f'Parallel:   {parallel_time:.1f}s')
print(f'Speedup:    {sequential_time / parallel_time:.1f}x')